# Forest Fire Detection — YOLOv11 Training
**Author:** Alaaeddine Bouchamla — ENISO M1 Telecommunications

### Before running
1. **Add dataset:** Right panel → **Data** → **Add Data** → search `smoke-fire-detection-yolo` by sayedgamal99 → Add
2. **Enable GPU:** Right panel → **Session options** → Accelerator → **GPU T4 x2**
3. **Enable Internet:** Right panel → Session options → Internet → **On**
4. Click **Run All** — you can close the tab, it keeps running

Estimated time: ~1.5 h (YOLOv11n) + ~4 h (YOLOv11x) on 2×T4

## 0. Install & verify

In [ ]:
!pip install ultralytics onnxruntime -q

import os
import torch
import ultralytics

ultralytics.checks()
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. Locate dataset & verify image counts

In [ ]:
# The dataset is already mounted by Kaggle — no download needed
DATASET_BASE = '/kaggle/input/smoke-fire-detection-yolo/data'

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_BASE, split, 'images')
    lbl_dir = os.path.join(DATASET_BASE, split, 'labels')
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    status = '✓' if n_img > 0 else '✗ MISSING'
    print(f'{status}  {split:6s}: {n_img:6d} images  {n_lbl:6d} labels')

# Expected: train=14122  val=3099  test=4306

## 2. Dataset config

In [ ]:
import yaml

dataset_config = {
    'path': DATASET_BASE,
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 2,
    'names': {0: 'fire', 1: 'smoke'}
}

CONFIG_PATH = '/kaggle/working/fire-dataset.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(dataset_config, f)

print(yaml.dump(dataset_config))

## 3. Train YOLOv11n — edge model
Uses both GPUs automatically when `device='0,1'`

In [ ]:
from ultralytics import YOLO

n_gpus = torch.cuda.device_count()
device = '0,1' if n_gpus >= 2 else '0'
print(f'Training on device: {device}')

model_n = YOLO('yolo11n.pt')

results_n = model_n.train(
    data=CONFIG_PATH,
    epochs=100,
    batch=32,           # doubled vs Colab — 2×T4 can handle it
    imgsz=640,
    device=device,
    project='/kaggle/working/runs',
    name='fire-yolo11n',
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=1.0,
    degrees=10.0,
    plots=True,
    save=True,
    exist_ok=True,
)

WEIGHTS_N = f'{results_n.save_dir}/weights/best.pt'
print(f'\nYOLOv11n complete → {WEIGHTS_N}')

## 4. Train YOLOv11x — cloud baseline

In [ ]:
model_x = YOLO('yolo11x.pt')

results_x = model_x.train(
    data=CONFIG_PATH,
    epochs=100,
    batch=16,           # yolo11x is heavier — 16 per GPU
    imgsz=640,
    device=device,
    project='/kaggle/working/runs',
    name='fire-yolo11x',
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=1.0,
    plots=True,
    save=True,
    exist_ok=True,
)

WEIGHTS_X = f'{results_x.save_dir}/weights/best.pt'
print(f'\nYOLOv11x complete → {WEIGHTS_X}')

## 5. Evaluate both models — paper metrics

In [ ]:
import json

def evaluate(weights, label):
    m = YOLO(weights).val(data=CONFIG_PATH, split='test', plots=True,
                          project='/kaggle/working/runs', name=f'eval-{label.split()[0].lower()}')
    return {
        'model':      label,
        'precision':  round(float(m.box.mp),    4),
        'recall':     round(float(m.box.mr),    4),
        'mAP50':      round(float(m.box.map50), 4),
        'mAP50_95':   round(float(m.box.map),   4),
        'per_class': {
            name: {
                'precision': round(float(p), 4),
                'recall':    round(float(r), 4),
                'mAP50':     round(float(a), 4),
            }
            for name, p, r, a in zip(
                m.names.values(), m.box.p, m.box.r, m.box.ap50
            )
        }
    }

print('Evaluating YOLOv11n...')
metrics_n = evaluate(WEIGHTS_N, 'YOLOv11n (edge, FP32)')
print('Evaluating YOLOv11x...')
metrics_x = evaluate(WEIGHTS_X, 'YOLOv11x (cloud)')

all_metrics = [metrics_n, metrics_x]

# ── Print table for paper ────────────────────────────────────────────
print('\n══ PAPER METRICS — Table I ══')
print(f'{"Model":<30} {"P":>6} {"R":>6} {"mAP50":>7} {"mAP50-95":>10}')
print('─' * 65)
for m in all_metrics:
    print(f"{m['model']:<30} {m['precision']:>6.4f} {m['recall']:>6.4f} "
          f"{m['mAP50']:>7.4f} {m['mAP50_95']:>10.4f}")
print(f"{'He et al. [5] (reference)':<30} {'0.949':>6} {'0.850':>6} {'0.901':>7} {'0.786':>10}")

print('\n══ PAPER METRICS — Table II (per-class mAP@0.5) ══')
print(f'{"Model":<30} {"fire":>8} {"smoke":>8}')
print('─' * 50)
for m in all_metrics:
    fire  = m['per_class'].get('fire',  {}).get('mAP50', 0)
    smoke = m['per_class'].get('smoke', {}).get('mAP50', 0)
    print(f"{m['model']:<30} {fire:>8.4f} {smoke:>8.4f}")

with open('/kaggle/working/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('\nSaved → /kaggle/working/metrics.json')

## 6. Export YOLOv11n to ONNX + benchmark CPU latency

In [ ]:
import time
import numpy as np
import onnxruntime as ort

onnx_path = YOLO(WEIGHTS_N).export(format='onnx', imgsz=640, simplify=True)
print(f'ONNX exported: {onnx_path}')

session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
inp = session.get_inputs()[0].name

for _ in range(10):   # warm-up
    session.run(None, {inp: dummy})

t0 = time.perf_counter()
for _ in range(100):
    session.run(None, {inp: dummy})
avg_ms = (time.perf_counter() - t0) / 100 * 1000

network_ms = 30
cloud_ms   = 50
render_ms  = 100
total_ms   = avg_ms + network_ms + cloud_ms + render_ms

print(f'\n══ LATENCY BREAKDOWN — Table III ══')
print(f'  Edge inference (ONNX, CPU):  {avg_ms:>7.1f} ms')
print(f'  4G/LTE uplink (estimated):   {network_ms:>7d} ms')
print(f'  Cloud DB write + WS push:    {cloud_ms:>7d} ms')
print(f'  Dashboard render:            {render_ms:>7d} ms')
print(f'  ─────────────────────────────────────────')
print(f'  Total end-to-end:            {total_ms:>7.0f} ms  ({total_ms/1000:.2f} s)')
print(f'  Target < 5 s  →  {"✓ PASS" if total_ms < 5000 else "✗ FAIL"}')

## 7. Copy ONNX model to output
Kaggle saves everything in `/kaggle/working/` automatically as notebook output.

In [ ]:
import shutil

out = '/kaggle/working'
shutil.copy(str(onnx_path), f'{out}/yolo11n-fire.onnx')
shutil.copy(WEIGHTS_N,      f'{out}/yolo11n-fire-best.pt')
shutil.copy(WEIGHTS_X,      f'{out}/yolo11x-fire-best.pt')

print('Output files:')
for f in sorted(os.listdir(out)):
    size_mb = os.path.getsize(os.path.join(out, f)) / 1e6
    print(f'  {f:<40} {size_mb:.1f} MB')

print('\nDone. Download outputs from the Kaggle notebook Output tab.')